<a href="https://colab.research.google.com/github/leman-cap13/NLP_projects/blob/main/DateDataGenerator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q datasets huggingface_hub pandas tqdm

In [ ]:
import random
import re

from datetime import date

import pandas as pd

from datasets import (
    Dataset,
    DatasetDict,
    Features,
    Value,
)

from huggingface_hub import (
    HfApi,
    login,
    whoami,
)

from tqdm.auto import tqdm

In [ ]:
SEED = 42


NUM_UNIQUE_DATES = 20_000
VARIANTS_PER_DATE = 5
TOTAL_ROWS = NUM_UNIQUE_DATES * VARIANTS_PER_DATE

assert TOTAL_ROWS == 100_000

rng = random.Random(SEED)

In [ ]:
START_DATE = date.min
END_DATE = date.max

print("Start:", START_DATE)
print("End:", END_DATE)
print("Total available dates:", END_DATE.toordinal())

In [ ]:
TOTAL_DATE_RANGE_DAYS = (END_DATE - START_DATE).days

In [ ]:
MIN_ORDINAL = START_DATE.toordinal()
MAX_ORDINAL = END_DATE.toordinal()

print(MIN_ORDINAL)
print(MAX_ORDINAL)

In [ ]:
MONTHS_FULL = [
    "january",
    "february",
    "march",
    "april",
    "may",
    "june",
    "july",
    "august",
    "september",
    "october",
    "november",
    "december",
]

MONTHS_SHORT = [
    "jan",
    "feb",
    "mar",
    "apr",
    "may",
    "jun",
    "jul",
    "aug",
    "sep",
    "oct",
    "nov",
    "dec",
]

WEEKDAYS_FULL = [
    "monday",
    "tuesday",
    "wednesday",
    "thursday",
    "friday",
    "saturday",
    "sunday",
]

WEEKDAYS_SHORT = [
    "mon",
    "tue",
    "wed",
    "thu",
    "fri",
    "sat",
    "sun",
]

In [ ]:
DAY_CARDINAL = {
    1: "one",
    2: "two",
    3: "three",
    4: "four",
    5: "five",
    6: "six",
    7: "seven",
    8: "eight",
    9: "nine",
    10: "ten",
    11: "eleven",
    12: "twelve",
    13: "thirteen",
    14: "fourteen",
    15: "fifteen",
    16: "sixteen",
    17: "seventeen",
    18: "eighteen",
    19: "nineteen",
    20: "twenty",
    21: "twenty one",
    22: "twenty two",
    23: "twenty three",
    24: "twenty four",
    25: "twenty five",
    26: "twenty six",
    27: "twenty seven",
    28: "twenty eight",
    29: "twenty nine",
    30: "thirty",
    31: "thirty one",
}

DAY_ORDINAL_WORD = {
    1: "first",
    2: "second",
    3: "third",
    4: "fourth",
    5: "fifth",
    6: "sixth",
    7: "seventh",
    8: "eighth",
    9: "ninth",
    10: "tenth",
    11: "eleventh",
    12: "twelfth",
    13: "thirteenth",
    14: "fourteenth",
    15: "fifteenth",
    16: "sixteenth",
    17: "seventeenth",
    18: "eighteenth",
    19: "nineteenth",
    20: "twentieth",
    21: "twenty first",
    22: "twenty second",
    23: "twenty third",
    24: "twenty fourth",
    25: "twenty fifth",
    26: "twenty sixth",
    27: "twenty seventh",
    28: "twenty eighth",
    29: "twenty ninth",
    30: "thirtieth",
    31: "thirty first",
}

In [ ]:
def ordinal(day: int) -> str:
    """
    Convert:
        1  -> 1st
        2  -> 2nd
        3  -> 3rd
        4  -> 4th
        11 -> 11th
        21 -> 21st
    """
    if 10 <= day % 100 <= 20:
        suffix = "th"
    else:
        suffix = {
            1: "st",
            2: "nd",
            3: "rd",
        }.get(day % 10, "th")

    return f"{day}{suffix}"

In [ ]:
for number in [1, 2, 3, 4, 11, 12, 13, 21, 22, 23, 31]:
    print(ordinal(number))

In [ ]:
FORMAT_NAMES = [
    # Month-name formats
    "month_full_day_year",
    "month_full_day_comma_year",
    "month_short_day_year",
    "month_short_dot_day_year",
    "day_month_full_year",
    "day_month_short_year",

    # Ordinal formats
    "ordinal_month_full_year",
    "ordinal_of_month_full_year",
    "the_ordinal_of_month_full_year",
    "month_full_ordinal_year",
    "month_full_the_ordinal_year",
    "day_of_month_full_year",

    # Natural-language formats
    "on_month_full_day_year",
    "on_day_month_full_year",
    "in_year_on_month_full_day",
    "month_full_day_in_year",
    "year_month_full_day",
    "year_month_short_day",
    "year_word_month_day",
    "year_comma_month_day",
    "year_month_day_words",
    "year_years_day_month_short",
    "year_yers_day_month_short",
    "year_yr_day_month_short",
    "day_month_short_year_word",
    "month_short_day_year_word",
    "year_day_month_labels",

    # Date sentence formats
    "date_is_month_day_year",
    "date_is_day_month_year",
    "date_colon_month_day_year",
    "calendar_date_day_month_year",

    # Weekday formats
    "weekday_month_day_year",
    "weekday_comma_month_day_year",
    "weekday_short_month_short_day_year",
    "weekday_day_month_year",
    "weekday_the_ordinal_month_year",
    "on_weekday_month_day_year",
    "weekday_short_day_month_short_year",

    # Numeric formats
    "mdy_slash_padded",
    "mdy_slash_unpadded",
    "mdy_dash_padded",
    "mdy_dash_unpadded",
    "dmy_dot_padded",
    "dmy_dot_unpadded",
    "eu_dmy_slash",
    "day_first_dmy_slash",
    "ymd_dash",
    "ymd_slash",
    "ymd_dot",
    "ymd_space",
    "ymd_compact",
    "us_mdy_compact",
    "eu_dmy_compact",

    # Explicit labels
    "month_eq_day_eq_year",
    "year_eq_month_eq_day",
    "short_labels",

    # Other separators
    "ymd_underscore",
    "mdy_underscore",
    "ymd_pipe",
    "mdy_pipe",

    # Month names with separators
    "day_slash_month_name_year",
    "month_name_slash_day_year",
    "year_slash_month_name_day",
    "day_dash_month_name_year",
    "month_name_dash_day_year",
    "year_dash_month_name_day",
    "day_dot_month_name_year",
    "month_name_dot_day_year",
    "year_dot_month_name_day",

    # Abbreviated and word-based forms
    "day_short_comma_year",
    "short_month_ordinal_year",
    "ordinal_short_month_year",
    "day_word_month_year",
    "ordinal_word_of_month_year",
    "month_ordinal_word_year",

    # Full sentence forms
    "the_date_month_day_year",
    "written_month_day_year",
    "year_first_sentence",
    "month_day_year_sentence",
    "day_month_year_sentence",
]

print("Number of formats:", len(FORMAT_NAMES))

assert len(FORMAT_NAMES) == 80

In [ ]:
def get_date_parts(d: date) -> dict:
    year = d.year
    month = d.month
    day = d.day

    return {
        "year": year,
        "year_4": f"{year:04d}",

        "month": month,
        "month_2": f"{month:02d}",
        "month_full": MONTHS_FULL[month - 1],
        "month_short": MONTHS_SHORT[month - 1],

        "day": day,
        "day_2": f"{day:02d}",
        "day_ordinal": ordinal(day),
        "day_cardinal_word": DAY_CARDINAL[day],
        "day_ordinal_word": DAY_ORDINAL_WORD[day],

        "weekday_full": WEEKDAYS_FULL[d.weekday()],
        "weekday_short": WEEKDAYS_SHORT[d.weekday()],
    }

In [ ]:
example_parts = get_date_parts(date(2020, 1, 7))
example_parts

In [ ]:
def render_clean_date(d: date, format_name: str) -> str:
    p = get_date_parts(d)

    y = p["year"]
    y4 = p["year_4"]

    m = p["month"]
    m2 = p["month_2"]

    day = p["day"]
    day2 = p["day_2"]

    month_full = p["month_full"]
    month_short = p["month_short"]

    weekday_full = p["weekday_full"]
    weekday_short = p["weekday_short"]

    day_ordinal = p["day_ordinal"]
    day_cardinal_word = p["day_cardinal_word"]
    day_ordinal_word = p["day_ordinal_word"]

    formats = {
        "month_full_day_year":
            f"{month_full} {day} {y}",

        "month_full_day_comma_year":
            f"{month_full} {day}, {y}",

        "month_short_day_year":
            f"{month_short} {day} {y}",

        "month_short_dot_day_year":
            f"{month_short}. {day} {y}",

        "day_month_full_year":
            f"{day} {month_full} {y}",

        "day_month_short_year":
            f"{day} {month_short} {y}",

        "ordinal_month_full_year":
            f"{day_ordinal} {month_full} {y}",

        "ordinal_of_month_full_year":
            f"{day_ordinal} of {month_full} {y}",

        "the_ordinal_of_month_full_year":
            f"the {day_ordinal} of {month_full} {y}",

        "month_full_ordinal_year":
            f"{month_full} {day_ordinal} {y}",

        "month_full_the_ordinal_year":
            f"{month_full} the {day_ordinal} {y}",

        "day_of_month_full_year":
            f"day {day} of {month_full} {y}",

        "on_month_full_day_year":
            f"on {month_full} {day} {y}",

        "on_day_month_full_year":
            f"on {day} {month_full} {y}",

        "in_year_on_month_full_day":
            f"in {y} on {month_full} {day}",

        "month_full_day_in_year":
            f"{month_full} {day} in {y}",

        "year_month_full_day":
            f"{y} {month_full} {day}",

        "year_month_short_day":
            f"{y} {month_short} {day}",

        "year_word_month_day":
            f"year {y} {month_full} {day}",

        "year_comma_month_day":
            f"{y}, {month_full} {day}",

        "year_month_day_words":
            f"{y} year {day} {month_short}",

        "year_years_day_month_short":
            f"{y} years {day} {month_short}",

        # Deliberately includes the user's typo
        "year_yers_day_month_short":
            f"{y} yers {day} {month_short}",

        "year_yr_day_month_short":
            f"{y} yr {day} {month_short}",

        "day_month_short_year_word":
            f"{day} {month_short} {y} year",

        "month_short_day_year_word":
            f"{month_short} {day} year {y}",

        "year_day_month_labels":
            f"year {y} day {day} month {month_short}",

        "date_is_month_day_year":
            f"date is {month_full} {day} {y}",

        "date_is_day_month_year":
            f"date is {day} {month_full} {y}",

        "date_colon_month_day_year":
            f"date: {month_full} {day}, {y}",

        "calendar_date_day_month_year":
            f"calendar date {day} {month_full} {y}",

        "weekday_month_day_year":
            f"{weekday_full} {month_full} {day} {y}",

        "weekday_comma_month_day_year":
            f"{weekday_full}, {month_full} {day}, {y}",

        "weekday_short_month_short_day_year":
            f"{weekday_short} {month_short} {day} {y}",

        "weekday_day_month_year":
            f"{weekday_full} {day} {month_full} {y}",

        "weekday_the_ordinal_month_year":
            f"{weekday_full} the {day_ordinal} of {month_full} {y}",

        "on_weekday_month_day_year":
            f"on {weekday_full} {month_full} {day} {y}",

        "weekday_short_day_month_short_year":
            f"{weekday_short} {day} {month_short} {y}",

        # Plain slash dates are defined as month/day/year.
        "mdy_slash_padded":
            f"{m2}/{day2}/{y4}",

        "mdy_slash_unpadded":
            f"{m}/{day}/{y}",

        "mdy_dash_padded":
            f"{m2}-{day2}-{y4}",

        "mdy_dash_unpadded":
            f"{m}-{day}-{y}",

        # Plain dot dates are defined as day.month.year.
        "dmy_dot_padded":
            f"{day2}.{m2}.{y4}",

        "dmy_dot_unpadded":
            f"{day}.{m}.{y}",

        # Explicit labels remove slash ambiguity.
        "eu_dmy_slash":
            f"eu {day2}/{m2}/{y4}",

        "day_first_dmy_slash":
            f"day-first {day2}/{m2}/{y4}",

        "ymd_dash":
            f"{y4}-{m2}-{day2}",

        "ymd_slash":
            f"{y4}/{m2}/{day2}",

        "ymd_dot":
            f"{y4}.{m2}.{day2}",

        "ymd_space":
            f"{y4} {m2} {day2}",

        "ymd_compact":
            f"{y4}{m2}{day2}",

        "us_mdy_compact":
            f"us {m2}{day2}{y4}",

        "eu_dmy_compact":
            f"eu {day2}{m2}{y4}",

        "month_eq_day_eq_year":
            f"month={m} day={day} year={y}",

        "year_eq_month_eq_day":
            f"year={y} month={m} day={day}",

        "short_labels":
            f"y:{y} m:{m2} d:{day2}",

        "ymd_underscore":
            f"{y4}_{m2}_{day2}",

        "mdy_underscore":
            f"{m2}_{day2}_{y4}",

        "ymd_pipe":
            f"{y4}|{m2}|{day2}",

        "mdy_pipe":
            f"{m2}|{day2}|{y4}",

        "day_slash_month_name_year":
            f"{day}/{month_full}/{y}",

        "month_name_slash_day_year":
            f"{month_full}/{day}/{y}",

        "year_slash_month_name_day":
            f"{y}/{month_full}/{day}",

        "day_dash_month_name_year":
            f"{day}-{month_full}-{y}",

        "month_name_dash_day_year":
            f"{month_full}-{day}-{y}",

        "year_dash_month_name_day":
            f"{y}-{month_full}-{day}",

        "day_dot_month_name_year":
            f"{day}.{month_full}.{y}",

        "month_name_dot_day_year":
            f"{month_full}.{day}.{y}",

        "year_dot_month_name_day":
            f"{y}.{month_full}.{day}",

        "day_short_comma_year":
            f"{day} {month_short}, {y}",

        "short_month_ordinal_year":
            f"{month_short} {day_ordinal} {y}",

        "ordinal_short_month_year":
            f"{day_ordinal} {month_short} {y}",

        "day_word_month_year":
            f"{day_cardinal_word} {month_full} {y}",

        "ordinal_word_of_month_year":
            f"{day_ordinal_word} of {month_full} {y}",

        "month_ordinal_word_year":
            f"{month_full} {day_ordinal_word} {y}",

        "the_date_month_day_year":
            f"the date is {month_full} {day} {y}",

        "written_month_day_year":
            f"written as {month_full} {day} {y}",

        "year_first_sentence":
            (
                f"the year is {y}, "
                f"the month is {month_full}, "
                f"the day is {day}"
            ),

        "month_day_year_sentence":
            f"month {month_full}, day {day}, year {y}",

        "day_month_year_sentence":
            f"day {day}, month {month_full}, year {y}",
    }

    if format_name not in formats:
        raise ValueError(f"Unknown format: {format_name}")

    return formats[format_name]

In [ ]:
test_date = date(2020, 1, 7)

for format_name in FORMAT_NAMES:
    source = render_clean_date(test_date, format_name)
    print(f"{format_name:40} -> {source}")

In [ ]:
MONTH_TYPOS = {
    "january": [
        "janurary",
        "januarry",
        "janury",
    ],
    "february": [
        "febuary",
        "febrary",
        "februarry",
    ],
    "march": [
        "marhc",
        "mrach",
    ],
    "april": [
        "aprl",
        "aripl",
    ],
    "may": [
        "maay",
    ],
    "june": [
        "junee",
        "jeun",
    ],
    "july": [
        "jully",
        "jluy",
    ],
    "august": [
        "agust",
        "augst",
    ],
    "september": [
        "septembar",
        "sepetember",
    ],
    "october": [
        "octomber",
        "octber",
    ],
    "november": [
        "novembar",
        "novmber",
    ],
    "december": [
        "decembar",
        "decemeber",
    ],
}

WORD_TYPOS = {
    "year": [
        "yer",
        "yera",
        "yr",
        "yers",
    ],
    "years": [
        "yers",
        "yaers",
    ],
    "month": [
        "mnth",
        "mont",
    ],
    "day": [
        "dy",
        "dya",
    ],
    "the": [
        "teh",
    ],
    "of": [
        "ov",
    ],
    "date": [
        "dtae",
    ],
}

In [ ]:
def replace_complete_word(
    text: str,
    original: str,
    replacement: str,
) -> str:
    pattern = rf"\b{re.escape(original)}\b"

    return re.sub(
        pattern,
        replacement,
        text,
        flags=re.IGNORECASE,
    )

In [ ]:
def apply_noise(
    clean_source: str,
    rng: random.Random,
) -> tuple[str, str]:
    """
    Add controlled noise without modifying date digits.

    Returns:
        noisy_source
        noise_description
    """
    text = clean_source
    noise_types = []

    # Occasionally misspell a full month name.
    if rng.random() < 0.12:
        for correct_month, typo_options in MONTH_TYPOS.items():
            pattern = rf"\b{correct_month}\b"

            if re.search(pattern, text, flags=re.IGNORECASE):
                replacement = rng.choice(typo_options)

                text = replace_complete_word(
                    text,
                    correct_month,
                    replacement,
                )

                noise_types.append("month_typo")
                break

    # Occasionally misspell one common date-related word.
    if rng.random() < 0.08:
        possible_words = [
            word
            for word in WORD_TYPOS
            if re.search(
                rf"\b{word}\b",
                text,
                flags=re.IGNORECASE,
            )
        ]

        if possible_words:
            selected_word = rng.choice(possible_words)
            replacement = rng.choice(
                WORD_TYPOS[selected_word]
            )

            text = replace_complete_word(
                text,
                selected_word,
                replacement,
            )

            noise_types.append("word_typo")

    # Remove commas or introduce one extra comma.
    if rng.random() < 0.08:
        if "," in text:
            text = text.replace(",", "")
            noise_types.append("punctuation_removed")

        elif " " in text:
            first_space = text.find(" ")

            text = (
                text[:first_space]
                + ", "
                + text[first_space + 1:]
            )

            noise_types.append("punctuation_added")

    # Replace ordinary spaces with two or three spaces.
    if rng.random() < 0.10:
        text = re.sub(
            r" ",
            lambda _: " " * rng.randint(2, 3),
            text,
        )

        noise_types.append("extra_spaces")

    # Apply case variation.
    case_type = rng.choices(
        population=[
            "lower",
            "title",
            "upper",
            "unchanged",
        ],
        weights=[
            55,
            20,
            10,
            15,
        ],
        k=1,
    )[0]

    if case_type == "lower":
        text = text.lower()

    elif case_type == "title":
        text = text.title()

    elif case_type == "upper":
        text = text.upper()

    if case_type != "unchanged":
        noise_types.append(f"case_{case_type}")

    # Occasionally include leading or trailing spaces.
    if rng.random() < 0.04:
        left_spaces = " " * rng.randint(1, 2)
        right_spaces = " " * rng.randint(1, 2)

        text = left_spaces + text + right_spaces

        noise_types.append("outer_spaces")

    if not noise_types:
        noise_description = "clean"
    else:
        noise_description = "+".join(noise_types)

    return text, noise_description

In [ ]:
def normalize_source_for_duplicate_check(text: str) -> str:
    text = text.strip().lower()
    text = re.sub(r"\s+", " ", text)

    return text

In [ ]:
FORCED_ROWS = {
    date(2020, 1, 7): [
        (
            "january 7 2020",
            "forced_month_day_year",
        ),
        (
            "7th of january 2020",
            "forced_ordinal_of_month",
        ),
        (
            "tuesday january 7 2020",
            "forced_weekday_month_day_year",
        ),
        (
            "01/07/2020",
            "forced_mdy_slash",
        ),
        (
            "2020-01-07",
            "forced_iso",
        ),
    ],

    date(2000, 1, 1): [
        (
            "2000 years 1 jan",
            "forced_years_typing",
        ),
        (
            "2000 yers 1 jan",
            "forced_yers_typo",
        ),
        (
            "year 2000 1 jan",
            "forced_year_first",
        ),
        (
            "1 jan 2000",
            "forced_day_month_year",
        ),
        (
            "2000-01-01",
            "forced_iso",
        ),
    ],
}

In [ ]:
FORCED_DATES = [
    date.min,
    date.max,

    date(2000, 1, 1),
    date(2020, 1, 7),

    date(400, 2, 29),
    date(2000, 2, 29),
    date(2024, 2, 29),
    date(2400, 2, 29),
]

In [ ]:
def choose_unique_dates(
    number_of_dates: int,
    rng: random.Random,
) -> list[date]:

    selected_ordinals = {
        forced_date.toordinal()
        for forced_date in FORCED_DATES
    }

    while len(selected_ordinals) < number_of_dates:
        random_ordinal = rng.randint(
            MIN_ORDINAL,
            MAX_ORDINAL,
        )

        selected_ordinals.add(random_ordinal)

    selected_dates = [
        date.fromordinal(ordinal_value)
        for ordinal_value in selected_ordinals
    ]

    rng.shuffle(selected_dates)

    return selected_dates

In [ ]:
def generate_rows_for_date(
    d: date,
    rng: random.Random,
    variants_per_date: int = 5,
) -> list[dict]:

    target = d.isoformat()

    rows = []
    seen_sources = set()

    # First add manually required examples.
    if d in FORCED_ROWS:
        for source, format_name in FORCED_ROWS[d]:
            normalized_source = (
                normalize_source_for_duplicate_check(source)
            )

            if normalized_source in seen_sources:
                continue

            seen_sources.add(normalized_source)

            rows.append({
                "source": source,
                "source_clean": source,
                "target": target,
                "format_name": format_name,
                "noise": "clean",
                "year": d.year,
                "month": d.month,
                "day": d.day,
                "weekday": WEEKDAYS_FULL[d.weekday()],
            })

    candidate_formats = FORMAT_NAMES.copy()
    rng.shuffle(candidate_formats)

    for format_name in candidate_formats:
        if len(rows) >= variants_per_date:
            break

        clean_source = render_clean_date(
            d,
            format_name,
        )

        source, noise_description = apply_noise(
            clean_source,
            rng,
        )

        normalized_source = (
            normalize_source_for_duplicate_check(source)
        )

        # Retry without noise if the noisy version duplicates
        # another source for the same date.
        if normalized_source in seen_sources:
            source = clean_source
            noise_description = "clean"

            normalized_source = (
                normalize_source_for_duplicate_check(source)
            )

        if normalized_source in seen_sources:
            continue

        seen_sources.add(normalized_source)

        rows.append({
            "source": source,
            "source_clean": clean_source,
            "target": target,
            "format_name": format_name,
            "noise": noise_description,
            "year": d.year,
            "month": d.month,
            "day": d.day,
            "weekday": WEEKDAYS_FULL[d.weekday()],
        })

    if len(rows) != variants_per_date:
        raise RuntimeError(
            f"Could not create {variants_per_date} "
            f"unique variants for {d}. "
            f"Created only {len(rows)}."
        )

    return rows

In [ ]:
def generate_date_dataset(
    number_of_unique_dates: int = NUM_UNIQUE_DATES,
    variants_per_date: int = VARIANTS_PER_DATE,
    seed: int = SEED,
) -> pd.DataFrame:

    local_rng = random.Random(seed)

    selected_dates = choose_unique_dates(
        number_of_dates=number_of_unique_dates,
        rng=local_rng,
    )

    all_rows = []

    for current_date in tqdm(
        selected_dates,
        desc="Generating dates",
    ):
        date_rows = generate_rows_for_date(
            d=current_date,
            rng=local_rng,
            variants_per_date=variants_per_date,
        )

        all_rows.extend(date_rows)

    local_rng.shuffle(all_rows)

    dataframe = pd.DataFrame(all_rows)

    return dataframe

In [ ]:
df = generate_date_dataset()

print(df.shape)

display(df.head(20))

In [ ]:
print(df.columns.tolist())

In [ ]:
display(
    df[df["target"] == "2020-01-07"][
        [
            "source",
            "target",
            "format_name",
        ]
    ]
)

In [ ]:
display(
    df[df["target"] == "2000-01-01"][
        [
            "source",
            "target",
            "format_name",
        ]
    ]
)

In [ ]:
assert len(df) == 100_000

print("Row count is correct.")

In [ ]:
assert df["target"].nunique() == 20_000

print("Unique target count is correct.")

In [ ]:
variants_per_target = df.groupby("target").size()

assert variants_per_target.min() == 5
assert variants_per_target.max() == 5

print("Every date has exactly five variants.")

In [ ]:
target_pattern = r"^\d{4}-\d{2}-\d{2}$"

valid_target_mask = df["target"].str.match(
    target_pattern
)

assert valid_target_mask.all()

print("All target formats are valid.")

In [ ]:
def reconstruct_target(row) -> str:
    reconstructed_date = date(
        int(row["year"]),
        int(row["month"]),
        int(row["day"]),
    )

    return reconstructed_date.isoformat()

In [ ]:
reconstructed_targets = df.apply(
    reconstruct_target,
    axis=1,
)

assert (
    reconstructed_targets == df["target"]
).all()

print("Every target matches its year, month and day.")

In [ ]:
exact_duplicate_count = df.duplicated(
    subset=[
        "source",
        "target",
    ]
).sum()

print("Exact duplicate count:", exact_duplicate_count)

assert exact_duplicate_count == 0

In [ ]:
normalized_sources = df["source"].map(
    normalize_source_for_duplicate_check
)

normalized_duplicate_count = (
    normalized_sources.duplicated().sum()
)

print(
    "Normalized duplicate source count:",
    normalized_duplicate_count,
)

assert normalized_duplicate_count == 0

In [ ]:
print("Minimum generated year:", df["year"].min())
print("Maximum generated year:", df["year"].max())

assert df["year"].min() == 1
assert df["year"].max() == 9999

In [ ]:
format_distribution = (
    df["format_name"]
    .value_counts()
    .sort_index()
)

display(format_distribution)

In [ ]:
unique_targets = df["target"].drop_duplicates().tolist()

split_rng = random.Random(SEED)
split_rng.shuffle(unique_targets)

train_target_count = 16_000
validation_target_count = 2_000

train_targets = set(
    unique_targets[:train_target_count]
)

validation_targets = set(
    unique_targets[
        train_target_count:
        train_target_count + validation_target_count
    ]
)

test_targets = set(
    unique_targets[
        train_target_count + validation_target_count:
    ]
)

In [ ]:
train_df = (
    df[df["target"].isin(train_targets)]
    .reset_index(drop=True)
)

validation_df = (
    df[df["target"].isin(validation_targets)]
    .reset_index(drop=True)
)

test_df = (
    df[df["target"].isin(test_targets)]
    .reset_index(drop=True)
)

In [ ]:
print("Train rows:", len(train_df))
print("Validation rows:", len(validation_df))
print("Test rows:", len(test_df))

In [ ]:
assert set(train_df["target"]).isdisjoint(
    set(validation_df["target"])
)

assert set(train_df["target"]).isdisjoint(
    set(test_df["target"])
)

assert set(validation_df["target"]).isdisjoint(
    set(test_df["target"])
)

print("No date leakage between splits.")

In [ ]:
ALL_PATH = "/content/date_converter_100k.csv"
TRAIN_PATH = "/content/train.csv"
VALIDATION_PATH = "/content/validation.csv"
TEST_PATH = "/content/test.csv"

In [ ]:
df.to_csv(
    ALL_PATH,
    index=False,
)

train_df.to_csv(
    TRAIN_PATH,
    index=False,
)

validation_df.to_csv(
    VALIDATION_PATH,
    index=False,
)

test_df.to_csv(
    TEST_PATH,
    index=False,
)

print("Saved:")
print(ALL_PATH)
print(TRAIN_PATH)
print(VALIDATION_PATH)
print(TEST_PATH)

In [ ]:
import os

print(os.path.exists("/content/train.csv"))
print(os.path.exists("/content/validation.csv"))
print(os.path.exists("/content/test.csv"))

In [ ]:
features = Features({
    "source": Value("string"),
    "source_clean": Value("string"),
    "target": Value("string"),
    "format_name": Value("string"),
    "noise": Value("string"),
    "year": Value("int32"),
    "month": Value("int8"),
    "day": Value("int8"),
    "weekday": Value("string"),
})

In [ ]:
train_dataset = Dataset.from_pandas(
    train_df,
    features=features,
    preserve_index=False,
)

validation_dataset = Dataset.from_pandas(
    validation_df,
    features=features,
    preserve_index=False,
)

test_dataset = Dataset.from_pandas(
    test_df,
    features=features,
    preserve_index=False,
)

In [ ]:
dataset = DatasetDict({
    "train": train_dataset,
    "validation": validation_dataset,
    "test": test_dataset,
})

In [ ]:
print(dataset)

In [ ]:
print(dataset["train"][0])

In [ ]:
!pip install -q -U datasets huggingface_hub

In [ ]:
!pip uninstall -y datasets

In [ ]:
!pip install --no-cache-dir --force-reinstall datasets huggingface_hub pyarrow

In [ ]:
import pandas as pd

train_df = pd.read_csv("/content/train.csv")
validation_df = pd.read_csv("/content/validation.csv")
test_df = pd.read_csv("/content/test.csv")

In [ ]:
from datasets import Dataset, DatasetDict

dataset = DatasetDict({
    "train": Dataset.from_pandas(
        train_df,
        preserve_index=False
    ),
    "validation": Dataset.from_pandas(
        validation_df,
        preserve_index=False
    ),
    "test": Dataset.from_pandas(
        test_df,
        preserve_index=False
    ),
})

In [ ]:
from datasets import Dataset, DatasetDict
from huggingface_hub import notebook_login, whoami

In [ ]:
from getpass import getpass
from huggingface_hub import login

hf_token = getpass("Hugging Face tokenini daxil et: ")

login(token=hf_token)

In [ ]:
account = whoami()

print(account)

In [ ]:
HF_USERNAME = account["name"]

print("My username:", HF_USERNAME)

In [ ]:
DATASET_NAME = "date-converter-100k"

REPO_ID = f"{HF_USERNAME}/{DATASET_NAME}"

print(REPO_ID)

In [ ]:
dataset.push_to_hub(
    REPO_ID,
    private=False,
    commit_message=(
        "Upload 100k synthetic date conversion examples"
    ),
)